# 情感模型训练（仅 20% 数据集 · 5090 稳妥版）

按顺序逐格运行即可。

## 本 Notebook 只做一件事：用 **20% 子集** 训练情感模型

终端里先准备好代码与环境：

```bash
git clone <你的仓库URL>
cd RSA
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r ml/requirements-finetune.txt
```

**数据（必填）：** 将已是 **20% 规模**（相对全量分层抽样）的 `train.csv`、`val.csv`、`test.csv` 上传到：

`ml/sentiment/data/splits_20pct/`

每份 CSV 需含列：`analysis_input_en`、`label_sentiment`（标签为 0/1/2）。不要把**全量**切分文件放进该目录，否则训练会用全量数据。

若你只有全量 `ml/sentiment/data/splits/{train,val,test}.csv`，请先在终端运行 `subset_splits.py --fraction 0.2` 生成 `splits_20pct` 后再用本 Notebook。

防断连（可选）：`tmux new -s train`，训练中 `Ctrl+B` 再按 `D` detach；恢复：`tmux attach -t train`。

In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    raise RuntimeError("请在仓库根目录打开并运行此 Notebook")

PYTHON_BIN = Path(".venv/bin/python")
if not PYTHON_BIN.exists():
    raise RuntimeError("未找到 .venv/bin/python，请先创建虚拟环境并安装依赖")

DIR_20 = ROOT / "ml/sentiment/data/splits_20pct"
for name in ("train.csv", "val.csv", "test.csv"):
    p = DIR_20 / name
    if not p.is_file():
        raise FileNotFoundError(
            f"本 Notebook 仅训练 20% 子集。请将三份 CSV 放到:\n  {DIR_20}/\n缺失: {name}"
        )

CFG_SAFE = ROOT / "ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml"
if not CFG_SAFE.exists():
    raise RuntimeError("未找到配置: ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml")

print("ROOT:", ROOT)
print("PYTHON_BIN:", PYTHON_BIN)
print("数据目录 (20%):", DIR_20)
print("训练配置:", CFG_SAFE)

In [ ]:
def run(cmd: list[str]):
    print("\n$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=ROOT)
    if p.returncode != 0:
        raise RuntimeError(f"命令失败: {' '.join(cmd)}")

## 0) GPU 环境自检（推荐）

确认云端已识别 GPU，并检查 PyTorch CUDA 可用性。

In [ ]:
run(["bash", "-lc", "nvidia-smi || true"])

run([
    str(PYTHON_BIN),
    "-c",
    "import torch; print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('device_count', torch.cuda.device_count())",
])

## 1) 数据行数（可选）

确认读入的是 20% 子集规模（行数应明显小于全量）。

In [ ]:
run([
    "bash",
    "-lc",
    "wc -l ml/sentiment/data/splits_20pct/train.csv ml/sentiment/data/splits_20pct/val.csv ml/sentiment/data/splits_20pct/test.csv",
])

## 2) 开始训练（稳妥版，仅读取 `splits_20pct`）

In [ ]:
run([
    str(PYTHON_BIN),
    "ml/sentiment/scripts/train_sentiment.py",
    "--config", "ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml",
    "--tokenized-cache-dir", "ml/sentiment/data/tokenized_cache_20pct",
])

## 3) 查看结果位置

In [ ]:
model_dir = ROOT / "models/sentiment/roberta-sentiment-20pct-gpu5090-safe-v0"
report_path = ROOT / "ml/sentiment/reports/sentiment_eval_20pct_gpu5090_safe_v0.json"

print("模型目录:", model_dir)
print("存在:", model_dir.exists())
print("评估报告:", report_path)
print("存在:", report_path.exists())

if report_path.exists():
    metrics = json.loads(report_path.read_text(encoding="utf-8"))
    print("\n关键指标:")
    for k in ("eval_loss", "eval_accuracy", "eval_runtime", "eval_samples_per_second"):
        if k in metrics:
            print(f"- {k}: {metrics[k]}")
    print("\n完整 JSON:")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))